# Barrandodger Archive Extraction Machine

This notebook downloads PDFs from the public GitHub repository, extracts the actual page text, and attempts OCR on scanned pages. It does not change the repository and does not use a paid AI API.

## iPhone instructions
1. Open https://colab.research.google.com/ in Safari.
2. Choose File → Upload notebook and upload this `.ipynb` file.
3. Tap the play icon on the code cell and approve Google Drive access.
4. Wait for the ZIP download. Run the cell again to process the next batch.
5. Upload the ZIP bundles here so the document contents can be examined.

The output includes a PDF inventory, extraction manifest, and page-marked text files. Errors are recorded instead of silently ignored. Large repositories require repeated batches.


In [ ]:
# Barrandodger Archive Extraction Machine — run in Google Colab
# Public repository only; no GitHub connector, no paid AI API, no repository changes.
!pip -q install pymupdf requests pandas

import os, re, json, zipfile, requests, pandas as pd
from pathlib import Path
from datetime import datetime, timezone
import fitz

REPO = "wezzo72/Barrandodger"
BATCH_SIZE = 25  # PDFs per run; rerun this cell to process the next batch
OCR_ENABLED = True
OCR_MAX_PAGES_PER_PDF = 80
OCR_DPI = 180

# Save progress in Google Drive so it survives notebook restarts.
from google.colab import drive, files
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Barrandodger_Review_Machine')
PDF_DIR, TEXT_DIR, BUNDLE_DIR = ROOT/'pdfs', ROOT/'text', ROOT/'bundles'
for p in (ROOT, PDF_DIR, TEXT_DIR, BUNDLE_DIR): p.mkdir(parents=True, exist_ok=True)

session=requests.Session()
session.headers.update({'Accept':'application/vnd.github+json','User-Agent':'Barrandodger-Archive-Extraction-Machine'})
api=f'https://api.github.com/repos/{REPO}'
r=session.get(api,timeout=30); r.raise_for_status(); meta=r.json(); branch=meta.get('default_branch','main')
print('Repository:',REPO,'branch:',branch)
tree_url=f'https://api.github.com/repos/{REPO}/git/trees/{branch}?recursive=1'
tr=session.get(tree_url,timeout=90); tr.raise_for_status(); tree=tr.json()
if tree.get('truncated'):
    print('WARNING: GitHub tree was truncated. The inventory may not include every repository file.')
items=[x for x in tree.get('tree',[]) if x.get('type')=='blob' and x.get('path','').lower().endswith('.pdf')]
items.sort(key=lambda x:x['path'].lower())
print('PDFs found:',len(items))
def safe_name(s): return re.sub(r'[^A-Za-z0-9._ -]','_',s.replace('/','__'))[:220]
inventory=[{'path':x['path'],'git_blob_sha':x.get('sha',''),'size_bytes':x.get('size',0),'raw_url':f"https://raw.githubusercontent.com/{REPO}/{branch}/"+requests.utils.requote_uri(x['path'])} for x in items]
pd.DataFrame(inventory).to_csv(ROOT/'pdf_inventory.csv',index=False)
(ROOT/'pdf_inventory.json').write_text(json.dumps(inventory,indent=2),encoding='utf-8')

OCR_AVAILABLE=False
if OCR_ENABLED:
    try:
        import pytesseract
        from PIL import Image
        import shutil
        if not shutil.which('tesseract'):
            os.system('apt-get -qq update && apt-get -qq install -y tesseract-ocr')
        OCR_AVAILABLE=bool(shutil.which('tesseract'))
    except Exception as e: print('OCR setup unavailable:',repr(e))
print('OCR available:',OCR_AVAILABLE)

def extract_pdf(pdf_path,txt_path):
    meta={'pages':0,'ocr_pages':0,'text_chars':0,'status':'ok','error':''}
    try:
        doc=fitz.open(pdf_path); meta['pages']=len(doc); chunks=[]
        for i,page in enumerate(doc):
            text=page.get_text('text') or ''
            if len(text.strip())<35 and OCR_AVAILABLE and i< OCR_MAX_PAGES_PER_PDF:
                try:
                    pix=page.get_pixmap(dpi=OCR_DPI,alpha=False)
                    img=Image.frombytes('RGB',[pix.width,pix.height],pix.samples)
                    ocr=pytesseract.image_to_string(img)
                    if len(ocr.strip())>len(text.strip()): text=ocr; meta['ocr_pages']+=1
                except Exception as e: text += '\n[OCR ERROR: '+repr(e)+']'
            chunks.append(f'\n===== PAGE {i+1} =====\n{text}')
        full='\n'.join(chunks); meta['text_chars']=len(full)
        if meta['text_chars']<20: meta['status']='no_text_found'
        txt_path.write_text(full,encoding='utf-8',errors='replace'); doc.close()
    except Exception as e:
        meta['status']='error'; meta['error']=repr(e); txt_path.write_text('[EXTRACTION ERROR]\n'+repr(e),encoding='utf-8')
    return meta

cp=ROOT/'checkpoint.json'; manifest=ROOT/'extraction_manifest.csv'
try: completed=json.loads(cp.read_text(encoding='utf-8')) if cp.exists() else {}
except Exception: completed={}
try: rows=pd.read_csv(manifest).to_dict(orient='records') if manifest.exists() else []
except Exception: rows=[]
by_path={x.get('path'):x for x in rows}
pending=[x for x in items if x['path'] not in completed]
batch=pending[:BATCH_SIZE]
print('Already processed:',len(completed),'| pending:',len(pending),'| this batch:',len(batch))
for n,item in enumerate(batch,1):
    rel=item['path']; print(f'[{n}/{len(batch)}] {rel}')
    txt=TEXT_DIR/(safe_name(rel)+'.txt'); pdf=PDF_DIR/safe_name(rel)
    raw=f"https://raw.githubusercontent.com/{REPO}/{branch}/"+requests.utils.requote_uri(rel)
    try:
        if not pdf.exists() or pdf.stat().st_size==0:
            rr=session.get(raw,timeout=120); rr.raise_for_status(); pdf.write_bytes(rr.content)
        details=extract_pdf(pdf,txt)
        row={'path':rel,'git_blob_sha':item.get('sha',''),'size_bytes':item.get('size',0),'text_file':str(txt.relative_to(ROOT)),'raw_url':raw,'processed_utc':datetime.now(timezone.utc).isoformat(),**details}
    except Exception as e:
        row={'path':rel,'git_blob_sha':item.get('sha',''),'size_bytes':item.get('size',0),'text_file':'','raw_url':raw,'processed_utc':datetime.now(timezone.utc).isoformat(),'pages':0,'ocr_pages':0,'text_chars':0,'status':'download_error','error':repr(e)}
    completed[rel]=row; by_path[rel]=row
    cp.write_text(json.dumps(completed,indent=2),encoding='utf-8')
    pd.DataFrame(list(by_path.values())).to_csv(manifest,index=False)

# Bundle all extracted text so far. Page markers preserve citations to PDF pages.
rows=list(by_path.values()); pd.DataFrame(rows).to_csv(manifest,index=False)
bundle=BUNDLE_DIR/f'Barrandodger_Text_Bundle_{len(completed)}_of_{len(items)}.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(ROOT/'pdf_inventory.csv',arcname='pdf_inventory.csv')
    z.write(manifest,arcname='extraction_manifest.csv')
    for row in rows:
        rel=row.get('text_file','')
        if rel and (ROOT/rel).exists(): z.write(ROOT/rel,arcname='text/'+Path(rel).name)
print('\nBUNDLE READY:',bundle)
print('Processed:',len(completed),'of',len(items))
print('If not finished, run this cell again for the next batch. Download each bundle and upload it to ChatGPT for document analysis.')
files.download(str(bundle))
